# Experiments: train + evaluate + explain ONE model (Steps 4–6)

Every experiment becomes **one MLflow run** on DagsHub containing:
- the resolved config, all hyper-parameters, git commit and dataset digest
- GPU/CPU usage, and per-epoch loss, learning rate and validation AUROC for every label
- the best checkpoint and the PyTorch model
- the test evaluation: calibration, uncertainty band, figures and report
- the Grad-CAM localisation scores and gallery

**How to use:** set `EXPERIMENT` below, then **Save Version → Save & Run All**. Run one commit per experiment (each stays well under Kaggle's 12 h limit).

**Settings:** GPU T4 x2, Internet ON. **Inputs:** NIH Chest X-rays dataset + your Phase 1 notebook. **Secrets (required here):** `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`, attached to this notebook. DagsHub is what lets runs from different commits be compared.

| Experiment | What it tests | Approx. time on T4 |
|---|---|---|
| `densenet121_224` | baseline (CheXNet-style) | 2 h |
| `densenet121_320` | + resolution + modern recipe (cosine, label smoothing, EMA) | 4 h |
| `convnext_tiny_320` | modern CNN backbone | 4 h |
| `efficientnet_v2_s_320` | efficient backbone | 3–4 h |


In [ ]:
EXPERIMENT = "densenet121_320"      # <- change for each commit
REPO_URL   = "https://github.com/<your-username>/MedVision.git"
RUN_SMOKE  = True                    # 2-minute check before the long run

In [ ]:
!git clone -q $REPO_URL MedVision
%cd MedVision
!pip install -q monai pydicom tabulate mlflow
!git log -1 --oneline
!python scripts/train.py --list

In [ ]:
import os
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for k in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    os.environ[k] = s.get_secret(k)          # fails loudly if a secret is missing / not attached
print("MLflow ->", os.environ["MLFLOW_TRACKING_URI"])

## Link the Phase 1 output (unzips the image cache to /tmp)

In [ ]:
!python scripts/link_phase1.py

## Smoke test (a separate MLflow run tagged smoke_test=true; the comparison ignores it)

In [ ]:
if RUN_SMOKE:
    !python scripts/train.py --experiment $EXPERIMENT --smoke --epochs 1 --max-steps 20 --subset 640

## Step 4: train

In [ ]:
!python scripts/train.py --experiment $EXPERIMENT

## Step 5: evaluate (same MLflow run)

In [ ]:
!python scripts/evaluate.py --experiment $EXPERIMENT

## Step 6: Grad-CAM (same MLflow run)

In [ ]:
!python scripts/gradcam_eval.py --experiment $EXPERIMENT --split test

## Results

In [ ]:
from IPython.display import Image, Markdown, display
W = f"/kaggle/working/{EXPERIMENT}"
display(Image(f"{W}/checkpoints/training_curves.png"))
display(Markdown(open(f"{W}/eval/eval_report.md").read()))
for f in ["auroc_ci", "status_focus", "band_sweep", "reliability_focus"]:
    display(Image(f"{W}/eval/{f}.png"))
display(Markdown(open(f"{W}/explain/explain_report.md").read()))
display(Image(f"{W}/explain/gradcam_gallery.png"))
print(open(f"{W}/checkpoints/mlflow_run_id.txt").read())